In [ ]:
# !pip install ninja --break-system-packages

In [ ]:
# !pip install --upgrade transformers --break-system-packages

In [ ]:
# !pip install git+https://github.com/intel/auto-round.git --break-system-packages

In [ ]:
# !pip install git+https://github.com/sustcsonglin/flash-linear-attention.git --no-build-isolation --break-system-packages

In [ ]:
# !pip install git+https://github.com/Dao-AILab/causal-conv1d.git --no-build-isolation --break-system-packages

In [1]:
import os
import torch
from auto_round import AutoRound
from huggingface_hub import HfApi, create_repo, notebook_login, get_token
from transformers import AutoModelForImageTextToText, AutoProcessor

In [2]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [3]:
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


PyTorch Version: 2.11.0+cu128
CUDA Available: True
CUDA Version: 12.8
GPU Name: NVIDIA RTX 6000 Ada Generation
VRAM: 47.5 GB


In [4]:
notebook_login()

In [ ]:
MODEL_ID = "Qwen/Qwen3.5-4B"
HF_USER = "Vishva007"
OUTPUT_BASE_DIR = "./AutoRound"
LOCAL_PATH = "./local_model"

In [ ]:
!hf download $MODEL_ID --local-dir $LOCAL_PATH

In [6]:
from safetensors import safe_open
import os

for file in os.listdir(LOCAL_PATH):
    if file.endswith(".safetensors"):
        path = os.path.join(LOCAL_PATH, file)
        print(f"\nChecking {file}")

        with safe_open(path, framework="pt") as f:
            keys = list(f.keys())

            mtp_keys = [k for k in keys if "mtp" in k.lower()]
            for k in mtp_keys:
                print(k)


Checking model.safetensors-00001-of-00002.safetensors
mtp.layers.0.mlp.down_proj.weight
mtp.layers.0.mlp.gate_proj.weight
mtp.layers.0.mlp.up_proj.weight

Checking model.safetensors-00002-of-00002.safetensors
mtp.fc.weight
mtp.layers.0.input_layernorm.weight
mtp.layers.0.post_attention_layernorm.weight
mtp.layers.0.self_attn.k_norm.weight
mtp.layers.0.self_attn.k_proj.weight
mtp.layers.0.self_attn.o_proj.weight
mtp.layers.0.self_attn.q_norm.weight
mtp.layers.0.self_attn.q_proj.weight
mtp.layers.0.self_attn.v_proj.weight
mtp.norm.weight
mtp.pre_fc_norm_embedding.weight
mtp.pre_fc_norm_hidden.weight


In [7]:
model = AutoModelForImageTextToText.from_pretrained(
    LOCAL_PATH, 
    dtype=torch.bfloat16,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(LOCAL_PATH)

tokenizer = processor.tokenizer


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

In [8]:
model

Qwen3_5ForConditionalGeneration(
  (model): Qwen3_5Model(
    (visual): Qwen3_5VisionModel(
      (patch_embed): Qwen3_5VisionPatchEmbed(
        (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 1024)
      (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-23): 24 x Qwen3_5VisionBlock(
          (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (attn): Qwen3_5VisionAttention(
            (qkv): Linear(in_features=1024, out_features=3072, bias=True)
            (proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (mlp): Qwen3_5VisionMLP(
            (linear_fc1): Linear(in_features=1024, out_features=4096, bias=True)
            (linear_fc2): Linear(in_features=4096, out_features=1024, bias=True)
            (act_fn): GELUTanh()
          )
        )
      )
 

In [24]:
TUNING_CONFIG = {
    "group_size": 32,
    "sym": True,
    "iters": 1000,           # Changed from 800 → 1000 (best recipe)
    "nsamples": 512,         # Keep at 512 (best recipe)
    "batch_size": 2,
    "seqlen": 2048,
    "low_gpu_mem_usage": False,  # Keep on GPU for speed
    "enable_torch_compile": True,  # JIT acceleration
    "quant_nontext_module": False,  # Keep Vision Tower in FP16 (Crucial for VLM accuracy)
    "layer_config": {
        "mtp": {"data_type": "bfloat16"},
        "mtp.fc": {"data_type": "bfloat16"}
    },
    "enable_alg_ext": True,
}

In [25]:
def push_to_hub(local_dir, repo_name, token):
    """Creates repo and uploads folder to Hugging Face."""
    full_repo_id = f"{HF_USER}/{repo_name}"
    print(f"\n[Hub] Pushing {local_dir} to {full_repo_id}...")

    try:
        api = HfApi()
        create_repo(
            full_repo_id, repo_type="model", exist_ok=True, private=False, token=token
        )

        api.upload_folder(
            folder_path=local_dir, repo_id=full_repo_id, repo_type="model", token=token
        )
        print(f"[Hub] ✅ Successfully uploaded: https://huggingface.co/{full_repo_id}")
    except Exception as e:
        print(f"[Hub] ❌ Error uploading: {e}")

In [26]:
ar = AutoRound(
    model=model,
    tokenizer=tokenizer,
    processor=processor,
    scheme="W2A16",
    **TUNING_CONFIG,
)

2026-06-14 15:09:57 INFO entry.py L587: Using MLLM mode for multimodal model.
Quantizing model.language_model.layers.0:   0%|          | 0/32 [18:00<?, ?it/s]


In [27]:
# The files will exist side-by-side or merged in this folder.
ar.quantize_and_save(
    OUTPUT_BASE_DIR, format="auto_round,auto_gptq", inplace=True
)

2026-06-14 15:09:58 INFO quantizer.py L303: using algorithm extension for quantization.


2026-06-14 15:09:59 INFO data_driven.py L661: start to cache block inputs
2026-06-14 15:09:59 INFO mllm.py L83: Using MLLM template: qwen3_5
2026-06-14 15:09:59 INFO calib_dataset.py L977: Preprocessing calibration dataset in a subprocess to avoid memory leaks...
2026-06-14 15:10:10 INFO data_driven.py L684: caching done
Quantizing model.language_model.layers.0:   0%|          | 0/32 [00:03<?, ?it/s]quantized 8/8 layers in the block, loss iter 0: 0.000000 -> iter 1: 0.000000
2026-06-14 15:10:56 INFO device.py L1449: 'peak_ram': 35.46GB, 'peak_vram': 19.5GB
Quantizing model.language_model.layers.1:   3%|▎         | 1/32 [00:45<23:34, 45.63s/it]quantized 8/8 layers in the block, loss iter 0: 0.000000 -> iter 1: 0.000000
2026-06-14 15:11:43 INFO device.py L1449: 'peak_ram': 35.46GB, 'peak_vram': 24.42GB
Quantizing model.language_model.layers.2:   6%|▋         | 2/32 [01:32<23:10, 46.34s/it]quantized 8/8 layers in the block, loss iter 0: 0.000001 -> iter 8: 0.000001
2026-06-14 15:12:30 INF

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-06-14 15:36:01 INFO missing_tensors.py L336: Found 15 tensor(s) in the source checkpoint that are absent from the saved output (e.g., MTP parameters): mtp.fc, mtp.layers.0.input_layernorm, mtp.layers.0.mlp.down_proj, mtp.layers.0.mlp.gate_proj, mtp.layers.0.mlp.up_proj, mtp.layers.0.post_attention_layernorm, mtp.layers.0.self_attn.k_norm, mtp.layers.0.self_attn.k_proj, mtp.layers.0.self_attn.o_proj, mtp.layers.0.self_attn.q_norm, mtp.layers.0.self_attn.q_proj, mtp.layers.0.self_attn.v_proj, mtp.norm, mtp.pre_fc_norm_embedding, mtp.pre_fc_norm_hidden. Copying them now...

Loading missing tensors:   0%|          | 0/2 [00:00<?, ?shard/s]

Loading missing tensors:  50%|█████     | 1/2 [00:00<00:00,  1.47shard/s]

Loading missing tensors: 100%|██████████| 2/2 [00:01<00:00,  1.68shard/s]
2026-06-14 15:36:03 INFO missing_tensors.py L823: Processing config.json to update quantization_config for missing tensors...
2026-06-14 15:36:03 INFO missing_tensors.py L790: Updated extra_config for 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-06-14 15:36:26 INFO missing_tensors.py L336: Found 15 tensor(s) in the source checkpoint that are absent from the saved output (e.g., MTP parameters): mtp.fc, mtp.layers.0.input_layernorm, mtp.layers.0.mlp.down_proj, mtp.layers.0.mlp.gate_proj, mtp.layers.0.mlp.up_proj, mtp.layers.0.post_attention_layernorm, mtp.layers.0.self_attn.k_norm, mtp.layers.0.self_attn.k_proj, mtp.layers.0.self_attn.o_proj, mtp.layers.0.self_attn.q_norm, mtp.layers.0.self_attn.q_proj, mtp.layers.0.self_attn.v_proj, mtp.norm, mtp.pre_fc_norm_embedding, mtp.pre_fc_norm_hidden. Copying them now...

Loading missing tensors:   0%|          | 0/2 [00:00<?, ?shard/s]



Loading missing tensors: 100%|██████████| 2/2 [00:00<00:00, 202.30shard/s]
2026-06-14 15:36:26 INFO missing_tensors.py L476: Successfully wrote 15 missing tensor(s) to 'model_extra_tensors.safetensors' in ./AutoRound/local_model-w2g32/auto-gptq.
2026-06-14 15:36:26 INFO device.py L1449: 'peak_ram': 35.46GB, 'peak_vram': 25.0GB


(Qwen3_5ForConditionalGeneration(
   (model): Qwen3_5Model(
     (visual): Qwen3_5VisionModel(
       (patch_embed): Qwen3_5VisionPatchEmbed(
         (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
       )
       (pos_embed): Embedding(2304, 1024)
       (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
       (blocks): ModuleList(
         (0-23): 24 x Qwen3_5VisionBlock(
           (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
           (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
           (attn): Qwen3_5VisionAttention(
             (qkv): Linear(in_features=1024, out_features=3072, bias=True)
             (proj): Linear(in_features=1024, out_features=1024, bias=True)
           )
           (mlp): Qwen3_5VisionMLP(
             (linear_fc1): Linear(in_features=1024, out_features=4096, bias=True)
             (linear_fc2): Linear(in_features=4096, out_features=1024, bias=True)
             (act_fn): GELUTanh()
           

In [28]:
base_name = MODEL_ID.split("/")[-1]
hf_token = get_token()

In [29]:
if hf_token:
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-w2g32/auto-round-auto-gptq"), 
        f"{base_name}-W2A16-AutoRound", 
        hf_token)
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-w2g32/auto-gptq"), 
        f"{base_name}-W2A16-AutoRound-GPTQ",
        hf_token
    )
else:
    print("No Hugging Face token found. Skipping upload to hub.")


[Hub] Pushing ./AutoRound/local_model-w2g32/auto-round-auto-gptq to Vishva007/Qwen3.5-4B-W2A16-AutoRound...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

[Hub] ✅ Successfully uploaded: https://huggingface.co/Vishva007/Qwen3.5-4B-W2A16-AutoRound

[Hub] Pushing ./AutoRound/local_model-w2g32/auto-gptq to Vishva007/Qwen3.5-4B-W2A16-AutoRound-GPTQ...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

[Hub] ✅ Successfully uploaded: https://huggingface.co/Vishva007/Qwen3.5-4B-W2A16-AutoRound-GPTQ
